# LSTM Crop Recommendation Model Training & Evaluation

This notebook downloads historical farm sensor data from Supabase, processes it into 24-hour sequences, and trains an LSTM (Long Short-Term Memory) neural network to predict the ideal crop (Maize, Cassava, Rice) based on the soil and climate conditions.

This serves as the evaluation phase for Chapter 4 of the final year project report.

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from supabase import create_client
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import keras
from keras import layers, callbacks
import joblib

# Set Keras backend to torch (or tensorflow, depending on environment)
os.environ["KERAS_BACKEND"] = "torch"

print("Libraries loaded successfully!")

## 1. Data Acquisition from Supabase
Fetch the simulated historical telemetry from the `capstone_dataset` table. In our simulation, specific nodes were dedicated to specific crops, generating distinct environmental profiles.

In [ ]:
# Load environment variables from .env file
load_dotenv()

URL = os.environ.get("VITE_SUPABASE_URL")
KEY = os.environ.get("VITE_SUPABASE_ANON_KEY")

if not URL or not KEY:
    raise ValueError("Missing Supabase credentials in .env")

print("Connecting to Supabase...")
supabase = create_client(URL, KEY)

# Fetch up to 20,000 rows ordered chronologically
response = supabase.table("capstone_dataset").select("*").order("Timestamp").limit(20000).execute()

data = response.data
print(f"Successfully fetched {len(data)} records.")

# Convert to pandas DataFrame for easier exploration
df = pd.DataFrame(data)
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

# Display sample data for Chapter 4.2.1
display(df.head())

## 2. Target Label Mapping & Preprocessing
Since the data was generated with specific profiles per node, we will map the nodes back to the `Target_Crop` they represent. This will be the target label `y` for our LSTM classification.

In [ ]:
NODE_CROP_MAP = {
    "NODE_01": "Maize",
    "NODE_02": "Maize",
    "NODE_03": "Cassava",
    "NODE_04": "Cassava",
    "NODE_05": "Rice",
    "NODE_06": "Rice"
}

# Map Node_ID to the Crop label
df['Target_Crop'] = df['Node_ID'].map(NODE_CROP_MAP)

# Plot the distribution of features across crops
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
features_to_plot = ["Nitrogen_mg_k", "Phosphorus_m", "Potassium_mg_", "Moisture_%", "Temperature_C", "Humidity_%"]

for i, feature in enumerate(features_to_plot):
    row, col = i // 3, i % 3
    sns.boxplot(data=df, x='Target_Crop', y=feature, ax=axes[row, col], palette='Set2')
    axes[row, col].set_title(f'{feature} Distribution')
    
plt.tight_layout()
plt.savefig('feature_distribution.png', dpi=300)
plt.show()

## 3. Sequence Generation for LSTM
LSTMs require 3D input arrays: `(samples, time_steps, features)`. We will group the data by node, scale it, and slice it into rolling windows of 24 time steps.

In [ ]:
SEQUENCE_LENGTH = 24
FEATURES = features_to_plot

# Scale the features
scaler = MinMaxScaler()
df[FEATURES] = scaler.fit_transform(df[FEATURES])

# Encode the target labels (Maize -> 0, Cassava -> 1, Rice -> 2)
label_encoder = LabelEncoder()
df['Crop_Label'] = label_encoder.fit_transform(df['Target_Crop'])

# Generate Sequences
X_list, y_list = [], []

for node_id, group in df.groupby('Node_ID'):
    group = group.sort_values('Timestamp').reset_index(drop=True)
    feature_data = group[FEATURES].values
    label_data = group['Crop_Label'].values
    
    for i in range(len(group) - SEQUENCE_LENGTH):
        X_list.append(feature_data[i: i + SEQUENCE_LENGTH])
        # Predict the crop for the sequence (the label is the same for the whole node)
        y_list.append(label_data[i + SEQUENCE_LENGTH - 1])

X = np.array(X_list)
y = np.array(y_list)

print(f"Input Shape X: {X.shape} (Samples, Time Steps, Features)")
print(f"Target Shape y: {y.shape} (Samples,)")

# Split into Train (80%) and Test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training samples: {len(X_train)}, Testing samples: {len(X_test)}")

## 4. LSTM Model Architecture & Training
We build a multi-class classification LSTM using Keras, with a Softmax output layer matching the number of crops (3).

In [ ]:
NUM_CLASSES = len(label_encoder.classes_)

model = keras.Sequential([
    keras.Input(shape=(SEQUENCE_LENGTH, len(FEATURES))),
    layers.LSTM(64, return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(32, return_sequences=False),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# Callbacks for early stopping and saving best model
callbacks_list = [
    callbacks.EarlyStopping(patience=5, restore_best_weights=True)
]

# Train the model
history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=50,
    batch_size=32,
    callbacks=callbacks_list,
    verbose=1
)

## 5. Visualizing Training Results (For Chapter 4.2.2)
Plotting the training loss versus validation loss to prove the model learned without overfitting.

In [ ]:
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss Curve')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy Curve')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.savefig('training_history.png', dpi=300)
plt.show()

## 6. Evaluation & Confusion Matrix (For Chapter 4.2.3)
Testing the model on unseen data and plotting a confusion matrix to show exactly which crops the model predicts correctly.

In [ ]:
# Evaluate on test set
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

# Get predictions
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

# Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title('LSTM Crop Prediction Confusion Matrix')
plt.xlabel('Predicted Crop')
plt.ylabel('Actual Crop')
plt.savefig('confusion_matrix.png', dpi=300)
plt.show()

## 7. Save Model & Scaler for Backend Integration
Exporting the model weights and data scaler so the FastAPI backend can load them for live inference.

In [ ]:
import os
model_dir = os.path.join("backend", "ml")
os.makedirs(model_dir, exist_ok=True)

model.save(os.path.join(model_dir, "lstm_crop_model.keras"))
joblib.dump(scaler, os.path.join(model_dir, "scaler_crop.pkl"))

# Save the label mapping as JSON
label_map = {int(i): str(c) for i, c in enumerate(label_encoder.classes_)}
with open(os.path.join(model_dir, "crop_labels.json"), "w") as f:
    json.dump(label_map, f)

print("Model, scaler, and labels successfully exported to backend/ml/")